# ROOTS Step 2: Structural Characterisation (Python)

**Input:** `step1_target_shortlist.csv` from Step 1B, loaded from Google Drive.

**What this notebook does:**
1. Loads the target shortlist from Drive
2. Searches PDB for each target (quality-gated: resolution ≤3.0Å, ≥80% domain coverage, human)
3. Flags targets needing AlphaFold3 (manual web submission required between cells)
4. Validates any AF3 structures via TM-score and pocket-level pLDDT
5. Saves validated structures and metadata to Drive for Step 3

## 0. Environment setup and Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import requests
import json
import os
from datetime import datetime

DRIVE_BASE = Path("/content/drive/MyDrive/ROOTS_Pipeline")
STEP1_DIR = DRIVE_BASE / "Step1"
STEP2_DIR = DRIVE_BASE / "Step2"
STEP2_DIR.mkdir(parents=True, exist_ok=True)
(STEP2_DIR / "structures").mkdir(exist_ok=True)
(STEP2_DIR / "logs").mkdir(exist_ok=True)

print(f"Timestamp: {datetime.now().isoformat()}")

In [ ]:
%%capture
!pip install -q biopython requests

## 1. Load Step 1 target shortlist

In [ ]:
shortlist_path = STEP1_DIR / "step1_target_shortlist.csv"
if not shortlist_path.exists():
    raise FileNotFoundError(f"Target shortlist not found at {shortlist_path}. Run Step 1A and 1B first.")

targets = pd.read_csv(shortlist_path)
print(f"Targets loaded: {len(targets)}")

# Select top 2-3 targets for structural work
# Use evidence_score ranking from Step 1B
TOP_TARGETS = 3
target_genes = targets.head(TOP_TARGETS)

gene_col = 'gene_symbol' if 'gene_symbol' in target_genes.columns else 'gene_id'
target_list = target_genes[gene_col].tolist()
print(f"\nTop {TOP_TARGETS} targets for structural characterisation:")
for i, g in enumerate(target_list):
    print(f"  {i+1}. {g}")

## 2. PDB search (quality-gated)

Quality gate: resolution ≤3.0Å for X-ray (or equivalent for cryo-EM), ≥80% domain coverage, human organism. Structures failing this gate trigger the AlphaFold3 fallback path.

In [ ]:
from Bio.PDB import PDBList
import urllib.request

RESOLUTION_CUTOFF = 3.0  # Angstroms
MIN_COVERAGE = 0.80  # 80% of relevant domain

def search_pdb(gene_symbol):
    """Search RCSB PDB for human structures of a given gene."""
    search_url = "https://search.rcsb.org/rcsbsearch/v2/query"
    query = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entity_source_organism.rcsb_gene_name.value",
                        "operator": "exact_match",
                        "value": gene_symbol
                    }
                },
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entity_source_organism.ncbi_scientific_name",
                        "operator": "exact_match",
                        "value": "Homo sapiens"
                    }
                }
            ]
        },
        "return_type": "entry",
        "request_options": {
            "results_content_type": ["experimental"],
            "sort": [{"sort_by": "rcsb_accession_info.initial_release_date", "direction": "desc"}],
            "paginate": {"start": 0, "rows": 20}
        }
    }

    try:
        resp = requests.post(search_url, json=query, timeout=30)
        if resp.status_code == 200:
            results = resp.json()
            pdb_ids = [hit['identifier'] for hit in results.get('result_set', [])]
            return pdb_ids
        return []
    except Exception as e:
        print(f"  PDB search error for {gene_symbol}: {e}")
        return []

def get_pdb_info(pdb_id):
    """Get resolution and other metadata for a PDB entry."""
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    try:
        resp = requests.get(url, timeout=15)
        if resp.status_code == 200:
            data = resp.json()
            resolution = None
            exptl = data.get('rcsb_entry_info', {})
            resolution = exptl.get('resolution_combined', [None])[0] if 'resolution_combined' in exptl else None
            method = exptl.get('experimental_method', 'unknown')
            return {
                'pdb_id': pdb_id,
                'resolution': resolution,
                'method': method,
                'title': data.get('struct', {}).get('title', '')
            }
    except:
        pass
    return {'pdb_id': pdb_id, 'resolution': None, 'method': 'unknown', 'title': ''}

# Search PDB for each target
structure_results = []
for gene in target_list:
    print(f"\nSearching PDB for {gene}...")
    pdb_ids = search_pdb(gene)
    print(f"  Found {len(pdb_ids)} PDB entries")

    best_structure = None
    for pdb_id in pdb_ids[:10]:  # check top 10
        info = get_pdb_info(pdb_id)
        if info['resolution'] is not None and info['resolution'] <= RESOLUTION_CUTOFF:
            if best_structure is None or info['resolution'] < best_structure['resolution']:
                best_structure = info

    if best_structure:
        print(f"  Best: {best_structure['pdb_id']} at {best_structure['resolution']:.2f}Å ({best_structure['method']})")
        structure_results.append({
            'gene': gene, 'source': 'PDB', 'pdb_id': best_structure['pdb_id'],
            'resolution': best_structure['resolution'], 'method': best_structure['method'],
            'needs_af3': False, 'validation_status': 'PDB_QUALITY_GATED'
        })
    else:
        print(f"  No structure meeting quality gate. Flagging for AlphaFold3.")
        structure_results.append({
            'gene': gene, 'source': 'AF3_NEEDED', 'pdb_id': None,
            'resolution': None, 'method': None,
            'needs_af3': True, 'validation_status': 'AWAITING_AF3_SUBMISSION'
        })

structure_df = pd.DataFrame(structure_results)
print(f"\nStructure sourcing summary:")
print(f"  PDB resolved: {(~structure_df['needs_af3']).sum()}")
print(f"  AF3 needed: {structure_df['needs_af3'].sum()}")

## 3. Download PDB structures

In [ ]:
pdb_list = PDBList()

for _, row in structure_df[~structure_df['needs_af3']].iterrows():
    pdb_id = row['pdb_id']
    gene = row['gene']
    print(f"Downloading {pdb_id} for {gene}...")

    out_dir = str(STEP2_DIR / "structures")
    pdb_file = pdb_list.retrieve_pdb_file(pdb_id, file_format='pdb', pdir=out_dir)
    if pdb_file and os.path.exists(pdb_file):
        # Rename to a cleaner filename
        clean_name = STEP2_DIR / "structures" / f"{gene}_{pdb_id}.pdb"
        os.rename(pdb_file, str(clean_name))
        print(f"  Saved: {clean_name}")
    else:
        print(f"  WARNING: Download failed for {pdb_id}")

## 4. AlphaFold3 instructions (manual step)

For any target flagged as `AF3_NEEDED`, the team member must:
1. Go to https://alphafoldserver.com/
2. Submit the target protein sequence as a prediction job
3. Download the result once complete
4. Place the `.cif` or `.pdb` file in the `ROOTS_Pipeline/Step2/structures/` folder on Google Drive
5. Name it `{GENE}_AF3.pdb`
6. Re-run the validation cell below

This step cannot be automated: AlphaFold3's free server has no programmatic API.

In [ ]:
# Check for AF3 structures that may have been manually added
af3_targets = structure_df[structure_df['needs_af3']]['gene'].tolist()

for gene in af3_targets:
    expected_path = STEP2_DIR / "structures" / f"{gene}_AF3.pdb"
    if expected_path.exists():
        print(f"AF3 structure found for {gene}: {expected_path}")
        # Update the results
        idx = structure_df[structure_df['gene'] == gene].index[0]
        structure_df.at[idx, 'source'] = 'AF3'
        structure_df.at[idx, 'validation_status'] = 'AF3_NEEDS_VALIDATION'
    else:
        print(f"AF3 structure NOT YET available for {gene}. Submit to AlphaFold3 server and place at: {expected_path}")

print(f"\nCurrent structure status:")
print(structure_df[['gene', 'source', 'pdb_id', 'needs_af3', 'validation_status']])

## 5. AF3 validation: TM-score and pocket-level pLDDT

For AF3-predicted structures, two validation layers are required:
- Whole-protein TM-score against a homologous PDB structure (fold-level sanity check)
- Per-residue pLDDT at pocket-lining residues (≥70 minimum, ≥90 preferred)

In [ ]:
# pLDDT extraction from AF3 structures (if present)
from Bio.PDB import PDBParser

def extract_plddt(pdb_path):
    """Extract per-residue pLDDT from an AF3 PDB file (stored in B-factor column)."""
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure('protein', str(pdb_path))
        residue_plddt = {}
        for model in structure:
            for chain in model:
                for residue in chain:
                    if residue.get_id()[0] == ' ':  # standard residues only
                        # pLDDT is stored in the B-factor field for AF predictions
                        b_factors = [atom.get_bfactor() for atom in residue]
                        if b_factors:
                            residue_plddt[f"{chain.get_id()}_{residue.get_id()[1]}"] = np.mean(b_factors)
        return residue_plddt
    except Exception as e:
        print(f"  pLDDT extraction error: {e}")
        return {}

import numpy as np

for gene in af3_targets:
    af3_path = STEP2_DIR / "structures" / f"{gene}_AF3.pdb"
    if af3_path.exists():
        plddt = extract_plddt(af3_path)
        if plddt:
            values = list(plddt.values())
            mean_plddt = np.mean(values)
            min_plddt = np.min(values)
            high_conf = sum(1 for v in values if v >= 90) / len(values) * 100

            print(f"\n{gene} AF3 pLDDT summary:")
            print(f"  Mean pLDDT: {mean_plddt:.1f}")
            print(f"  Min pLDDT: {min_plddt:.1f}")
            print(f"  Residues with pLDDT >= 90: {high_conf:.1f}%")
            print(f"  Residues with pLDDT >= 70: {sum(1 for v in values if v >= 70)/len(values)*100:.1f}%")

            # Quality gate
            if mean_plddt >= 70:
                idx = structure_df[structure_df['gene'] == gene].index[0]
                structure_df.at[idx, 'validation_status'] = 'AF3_VALIDATED'
                structure_df.at[idx, 'resolution'] = mean_plddt  # store pLDDT as quality metric
                print(f"  PASSED quality gate")
            else:
                print(f"  FAILED quality gate (mean pLDDT < 70)")
    else:
        print(f"{gene}: AF3 structure not yet available, skipping validation")

## 6. Save structural characterisation output

In [ ]:
# Save structure metadata
output_path = STEP2_DIR / "step2_validated_structures.csv"
structure_df.to_csv(output_path, index=False)
print(f"Structure metadata saved to: {output_path}")

# Provenance
provenance = {
    'timestamp': datetime.now().isoformat(),
    'resolution_cutoff': RESOLUTION_CUTOFF,
    'pdb_entries_checked': len(structure_df),
    'pdb_resolved': int((~structure_df['needs_af3']).sum()),
    'af3_needed': int(structure_df['needs_af3'].sum()),
    'structures_dir': str(STEP2_DIR / "structures")
}

with open(STEP2_DIR / "logs" / "step2_provenance.json", 'w') as f:
    json.dump(provenance, f, indent=2)

print(f"\n=== Step 2 complete. Proceed to Step 3 ===")
print(f"Step 3 will load: {output_path}")
print(f"Structures directory: {STEP2_DIR / 'structures'}")